# Google Cloud Run 챗봇 배포 및 운영 실습

이 노트북은 **Google Cloud Run(완전 관리형 서버리스 컨테이너)** 환경에 Gemini 3.8 Flash & 3.7 Flash 웹 챗봇을 빌드하고 배포하는 전체 과정을 다룹니다.

### 🌟 Compute Engine 대비 Cloud Run의 장점
1. **Scale-to-Zero (비용 최적화)**: 유휴 시간(Idle)에 컨테이너가 0개로 축소되어 비용이 전혀 발생하지 않습니다. (Compute Engine은 VM과 디스크 비용이 24시간 발생)
2. **자동 HTTPS/TLS 인증서 관리**: 별도의 Nginx, Certbot, DDNS(sslip.io) 설정 없이 구글이 관리하는 공인 인증서가 기본 제공됩니다.
3. **초간편 배포 (Source-to-Cloud)**: `gcloud run deploy --source .` 명령 하나로 Cloud Build 컨테이너 빌드 및 배포가 일괄 처리됩니다.
4. **Google Secret Manager 기본 통합**: `--set-secrets` 플래그로 API 키를 안전하게 컨테이너 환경변수에 주입할 수 있습니다.

---

## 1. 사전 환경 확인 및 GCP 인증

현재 로그인된 계정과 활성 프로젝트를 확인합니다.

In [ ]:
import sys
import subprocess
IS_WIN = sys.platform == "win32"
import json

PROJECT_ID = "iceu-songpa21"
REGION = "us-central1"
SERVICE_NAME = "gemini-chatbot"
SECRET_NAME = "GEMINI_API_KEY"

# 1. gcloud 프로젝트 설정
subprocess.run(["gcloud", "config", "set", "project", PROJECT_ID], check=True, shell=IS_WIN)
print(f"✅ 활성 프로젝트 설정 완료: {PROJECT_ID}")

# 2. 인증 계정 확인
account = subprocess.check_output(["gcloud", "config", "get-value", "account"], shell=IS_WIN).decode().strip()
print(f"👤 현재 로그인 계정: {account}")

## 2. 필수 Google Cloud API 활성화

Cloud Run 서비스 구동에 필요한 API들을 일괄 활성화합니다:
- `run.googleapis.com`: Cloud Run 서비스
- `cloudbuild.googleapis.com`: 소스코드 기반 자동 컨테이너 이미지 빌드
- `secretmanager.googleapis.com`: Gemini API 키 보안 조회
- `artifactregistry.googleapis.com`: 빌드된 컨테이너 이미지 저장소

In [ ]:
apis = [
    "run.googleapis.com",
    "cloudbuild.googleapis.com",
    "secretmanager.googleapis.com",
    "artifactregistry.googleapis.com"
]
print(f"⏳ 필수 API 활성화 중... ({', '.join(apis)})")
subprocess.run(["gcloud", "services", "enable"] + apis, check=True, shell=IS_WIN)
print("✅ 필수 API 활성화 완료!")

## 3. Secret Manager 권한 및 키 검증

Cloud Run의 기본 서비스 계정이 Secret Manager에 접근할 수 있도록 권한(`roles/secretmanager.secretAccessor`)을 부여합니다.

In [ ]:
# 프로젝트 번호 및 서비스 계정 조회
pnum = subprocess.check_output(
    ["gcloud", "projects", "describe", PROJECT_ID, "--format=value(projectNumber)"], shell=IS_WIN
).decode().strip()
sa = f"{pnum}-compute@developer.gserviceaccount.com"

# Secret Accessor 권한 부여
subprocess.run([
    "gcloud", "projects", "add-iam-policy-binding", PROJECT_ID,
    f"--member=serviceAccount:{sa}",
    "--role=roles/secretmanager.secretAccessor"
], check=False, shell=IS_WIN)
print(f"🔐 서비스 계정 권한 바인딩 완료: {sa}")

## 4. Cloud Run 원클릭 소스 배포 (`gcloud run deploy`)

Dockerfile과 소스코드가 포함된 현재 디렉터리를 소스로 지정하여 Cloud Run에 배포합니다.
- `--allow-unauthenticated`: 웹 브라우저에서 누구나 접속할 수 있도록 공개 허용
- `--min-instances=0`: 유휴 상태 시 0개로 축소하여 비용 절감
- `--set-secrets`: Secret Manager의 GEMINI_API_KEY를 컨테이너 환경변수로 안전하게 연결

In [ ]:
import time

print(f"🚀 Cloud Run 서비스 배포를 시작합니다: {SERVICE_NAME} (Region: {REGION})")
start_time = time.time()

deploy_cmd = [
    "gcloud", "run", "deploy", SERVICE_NAME,
    "--source=.",
    f"--project={PROJECT_ID}",
    f"--region={REGION}",
    "--platform=managed",
    "--allow-unauthenticated",
    "--min-instances=0",
    "--max-instances=5",
    "--memory=1Gi",
    "--cpu=1",
    "--timeout=120",
    f"--set-secrets=GEMINI_API_KEY={SECRET_NAME}:latest"
]

res = subprocess.run(deploy_cmd, capture_output=True, text=True, shell=IS_WIN)
print(res.stdout)
if res.returncode != 0:
    print("❌ 배포 실패 stderr:", res.stderr)
else:
    elapsed = round(time.time() - start_time, 1)
    print(f"🎉 배포 성공! (소요 시간: {elapsed}초)")

## 5. 배포된 서비스 URL 확인 및 API 상태 검증

In [ ]:
import urllib.request
import json

# 1. 서비스 URL 조회
url_out = subprocess.check_output([
    "gcloud", "run", "services", "describe", SERVICE_NAME,
    f"--project={PROJECT_ID}",
    f"--region={REGION}",
    "--format=value(status.url)"
]).decode().strip()

print(f"🌐 배포된 Cloud Run 서비스 URL: {url_out}")

# 2. 헬스체크 엔드포인트 호출 (/health)
with urllib.request.urlopen(f"{url_out}/health") as resp:
    health_data = json.loads(resp.read().decode())
    print("🩺 헬스체크 응답:", health_data)

# 3. 상태 API 엔드포인트 호출 (/api/status)
with urllib.request.urlopen(f"{url_out}/api/status") as resp:
    status_data = json.loads(resp.read().decode())
    print("📊 서버 상태 응답:", status_data)

## 6. 챗봇 대화 테스트 (POST `/api/chat`)

배포된 Cloud Run 백엔드에 질문을 전송하고 Gemini 3.8 Flash 모델의 응답과 사고/도구 연동 결과를 확인합니다.

In [ ]:
test_payload = json.dumps({
    "input": "Google Cloud Run과 Compute Engine의 주요 차이점 3가지를 한국어로 핵심만 요약해줘.",
    "model": "gemini-3.8-flash"
}).encode("utf-8")

req = urllib.request.Request(
    f"{url_out}/api/chat",
    data=test_payload,
    headers={"Content-Type": "application/json"}
)

print("💬 챗봇에 테스트 질문 전송 중...")
try:
    with urllib.request.urlopen(req, timeout=70) as resp:
        chat_result = json.loads(resp.read().decode())
        print("\n🤖 [Gemini 응답]:\n")
        print(chat_result.get("reply"))
        print("\n🛠️ 사용 도구:", chat_result.get("toolsUsed"))
        print("💡 사고 과정 포함 여부:", chat_result.get("hasThought"))
except Exception as err:
    print("❌ 챗봇 호출 오류:", err)